# In-Class Activity #5 (July 21, 2026)

In [1]:
library(tidyverse)
library(moderndive)
library(car)

── Attaching core tidyverse packages ─────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ───────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: carData


Attaching package: ‘car’


The following object is masked from ‘package:dplyr’:

    recode


The following object is masked from ‘package:purrr’:

    some




In [2]:
wage_data <- read.table("data/wage.txt", header=TRUE)
head(wage_data)

,education,south,sex,experience,union,wage,age,race,occupation,sector,marr
,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>,<int>,<int>,<int>
1,8,0,1,21,0,5.10,35,2,6,1,1
2,9,0,1,42,0,4.95,57,3,6,1,1
3,12,0,0,1,0,6.67,19,3,6,1,0
4,12,0,0,4,0,4.00,22,3,6,0,0
5,12,0,0,17,0,7.50,35,3,6,0,1
6,13,0,0,9,1,13.07,28,3,6,0,0


# Part A

In [3]:
# wage_data %>%
#     select(-south, -sex, -union, -race, -occupation, -sector, -marr) %>%
#     cor()

wage_data %>%
    select(education, experience, age) %>%
    cor()

,education,experience,age
education,1.0000000,-0.3526764,-0.1500195
experience,-0.3526764,1.0000000,0.9779612
age,-0.1500195,0.9779612,1.0000000


Based on the pairwise correlations of the continuous variables, there appears to be strong positive correlation between `age` and `experience` which exhibit a correlation of `0.9779612`. There appears to also be moderate/weak negative correlation between `experience` and `education`: `0.3526764`.

In [4]:
# wage_model <- lm(wage ~ education + experience + wage + age, data = wage_data)
# vif(wage_model)

wage_model <- lm(wage ~ education + experience + age, data = wage_data)
vif(wage_model)

education experience        age 
  229.5738  5147.9190  4611.4008

In addition to looking at the pairwise correlation, I also observed the correlations using `vif` for a linear model trained on just the continuous numerical input variables. This indicated a very large gvif value for experience. Therefore, we can remove this predictor from the model and fit it again.

In [5]:
wage_model <- lm(wage ~ education + age, data = wage_data)
vif(wage_model)

education       age 
 1.023024  1.023024

The `vif` values for `education` and `age` are both below `sqrt(5)` and `5` which indicate that these do not exhibit much collinearity.

We can also train a model using the categorical variables.

In [6]:
wage_model <- lm(wage ~ education + experience + age + factor(occupation) + factor(race) + factor(sector) + south + sex + union + marr, data = wage_data)
vif(wage_model)

,GVIF,Df,GVIF^(1/(2*Df))
education,234.855660,1,15.325001
experience,5212.811724,1,72.199804
age,4669.808671,1,68.335998
factor(occupation),3.020968,5,1.116901
factor(race),1.094538,2,1.022840
factor(sector),1.444859,2,1.096368
south,1.061343,1,1.030215
sex,1.272000,1,1.127830
union,1.128802,1,1.062451
marr,1.111966,1,1.054498


In [7]:
wage_model <- lm(wage ~ education + age + factor(occupation) + factor(race) + factor(sector) + south + sex + union + marr, data = wage_data)
vif(wage_model)

,GVIF,Df,GVIF^(1/(2*Df))
education,1.813186,1,1.346546
age,1.190469,1,1.091086
factor(occupation),2.999691,5,1.116112
factor(race),1.094327,2,1.022791
factor(sector),1.444471,2,1.096294
south,1.060840,1,1.029971
sex,1.268426,1,1.126244
union,1.128733,1,1.062418
marr,1.109491,1,1.053324


In [8]:
sqrt(5)

[1] 2.236068

We can see that experience has the greatest generalized GVIF value when also accounting for the categorical variables, and after removing this value we can see that all GVIF values become significantly lower. All of the generalized GVIF values are now below sqrt(5).

# Part B

The `experience` predictor variable can be removed from the model because the gvif value of `experience` predictor is significantly above the `sqrt(5)` and `5` thresholds when we observed the vif values. Also, this was the continuous variable that exhibited collinearity when observing the pairwise correlation table.

In [9]:
wage_model <- lm(wage ~ education + age, data = wage_data)
vif(wage_model)

education       age 
 1.023024  1.023024

In [10]:
wage_model <- lm(wage ~ education + age + factor(occupation) + factor(race) + factor(sector) + south + sex + union + marr, data = wage_data)
vif(wage_model)

,GVIF,Df,GVIF^(1/(2*Df))
education,1.813186,1,1.346546
age,1.190469,1,1.091086
factor(occupation),2.999691,5,1.116112
factor(race),1.094327,2,1.022791
factor(sector),1.444471,2,1.096294
south,1.060840,1,1.029971
sex,1.268426,1,1.126244
union,1.128733,1,1.062418
marr,1.109491,1,1.053324


# Part C

This is an **observational study** so we therefore **cannot** claim a causal relationship between the significant predictors and the response. We know this because the different input variables are existing characteristics about the different individuals in the dataset rather than randomly assigned attributes. A random assignment of these values would not be possible (and/or unethical).